In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Dictionaries (Hash Tables)
## Econ 524


Motivating example:  lookup by value

Sometimes we want to be able to find the index that corresponds to a given value in the array.  For example, which index in the array s has the value “California”?

One way to do this is the nonzero function, which is O(n).  

If the values are unique (i.e. each value occurs at most once in the array), we can use hash tables (dictionaries in python) for this and lookup in O(1).

In this example, s is a string array with no repeated values.  We will use a dictionary to lookup the index given the string value.


### nonzero vs dictionary

np.nonzero(s=="California")

First, the expression s=="California" is evaluated, leading to a bool array.

Then, the nonzero function goes element-by-element through the bool array, noting the index of all True (i.e. 1) values.

Since it checks every element of the bool array, it is O(n) where n is the length of s.

If we create a dictionary that maps each value to the index, then we can lookup index in O(1).

### Lookup by value example

In [3]:
import numpy as np
s=np.array(["Arizona","Nevada","California","Oregon","Washington"])

d = dict()
for i in range(len(s)):
    d[s[i]]=i

d

s[2]
np.nonzero(s=="California")[0][0]  
#Why two [0]?  The first is because nonzero returns a tuple, so [0] yields the first tuple, 
#the second is because nonzero yields an array of indices that are non-zero, and we want the first (and only) such index
d["California"]

%timeit np.nonzero(s=="California")[0][0]
%timeit d["California"]


{'Arizona': 0, 'Nevada': 1, 'California': 2, 'Oregon': 3, 'Washington': 4}

'California'

2

2

8.74 μs ± 82.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
129 ns ± 1.6 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [22]:
#Another Example with a bigger array
import string
np.random.seed(0)
s=np.array([''.join(np.random.choice(list(string.ascii_lowercase)) for i in range(10)) for j in range(10000)])
len(np.unique(s))
d = dict()
for i in range(len(s)):
    d[s[i]]=i
s[5000]
np.nonzero(s=="cbuhafyvzh")[0][0]
d["cbuhafyvzh"]


10000

'cbuhafyvzh'

5000

5000

In [24]:
%timeit np.nonzero(s=="cbuhafyvzh")[0][0]
%timeit d["cbuhafyvzh"]

56.1 μs ± 527 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
127 ns ± 4.55 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


### What about the setup time?

The loop to create the dictionary is O(n), so did we really save any time?

Yes, if there are going to be a lot of lookups.  The alternative to dictionaries is using nonzero (or something similar) which is O(n) vs dictionary lookup which is O(1).

Setting up dictionary takes time O(n), so…
* If the number of lookups is O(1), then there is no advantage to a dictionary
* But if the number of lookups is a function of n, then dictionary may be worth it
* For example, if number of lookups is O(logn) then with dictionary total time is O(n)+O(logn)=O(n) and without dictionary total time is O(n)*O(logn)=O(nlogn)



### GPA example 

Suppose you grade rosters for several courses that include student name/id, grade, and number of units in each row.  You want to calculate each student's GPA.

Let's assume there are n students and each student has a constant number of courses so the size of the grade roster data frame is O(n)

You need to calculate a running total for each student of units and of grade\*units.

One approach would be do one student at a time.  For each student, go through grade rosters, pick the correct rows and compute the sums.  This would be very slow since you have to traverse the grade roster data frame multiple times (once per student).  O(n^2)

Another approach would be to traverse the data frame only once, and have a big if/elif statement to figure out which student a given row is and then add to that student's running sum.  The big if/elif statement would also be slow if there are a lot of students.  O(n^2)





In [14]:
# slow approach to calculating GPA
import numpy as np

def gpa_slow(ids,units,grades):
    uni=np.unique(ids)
    gpa=np.zeros(len(uni))
    for i in range(len(uni)):
        gradesum=np.sum(grades[ids==uni[i]]*units[ids==uni[i]])
        unitssum = np.sum(units[ids==uni[i]])
        gpa[i]=gradesum/unitssum
    r=(uni,gpa)
    return r

In [18]:
import pandas as pd
data = pd.read_csv("gradelist.csv")
gpa_slow(data.id,data.units,data.grade)

(array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30], dtype=int64),
 array([0.85185185, 1.85714286, 0.85714286, 1.84      , 3.11111111,
        2.07407407, 1.7826087 , 2.03703704, 1.53846154, 2.25925926,
        2.28571429, 1.28571429, 1.57142857, 2.25925926, 1.38461538,
        2.23076923, 2.28571429, 2.45833333, 2.65384615, 1.96296296,
        1.71428571, 1.61538462, 1.24      , 2.        , 2.14814815,
        2.07407407, 2.26923077, 1.85714286, 2.07692308, 1.29166667]))

### GPA example (with dictionary)

Dictionaries can improve on both of the slow approaches described above.

Algorithm:

1. Create two empty dictionaries:  one that maps student id to the units sum and another that maps student id to grade\*units sum.

2. For each row in data frame:  
* lookup student in both dictionaries - if student has a dictionary entry, then add the data from this row to units sum and also to the grade\*units sum.  if student does not have dictionary entry, then create it based on data from this row.  
   
3. For each student:
* calculate GPA by dividing values in the two dictionaries.
   
The body of both loops is O(1) so each loop is O(n).  So, the algorithm runs in O(n) + O(n) = O(n)

### GPA example (Pandas groupby)

There is a groupby function in Pandas that will be even faster than our own implementation!  More on this later.

### Comparing speed of 4 gpa functions

In [7]:
data = pd.read_csv("grades_large.csv")

%timeit gpa_groupby(data)


29.2 ms ± 2.4 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [8]:
%timeit gpa_series(data) 

7.59 s ± 449 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%timeit gpa_dict(data)

4.86 s ± 80.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
%timeit -n 1 gpa_slow(data)

3min 15s ± 3.48 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
